In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
nb.env_report()


# Test-split Bayes proxy sanity check

**Purpose.** Inspect a recorded synthetic project artifact through the project APIs and show what it supports. Source: `reports/is_high_risk_auc.json`. This notebook loads no external or real customer data.

**Interpretation limits.** Analytic label-generator Bayes proxy and federated model AUC on the same per-client test split. This is a sanity comparator, not a claim that the proxy is the true Bayes optimum for all populations.


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    mode: str = nb.MODE
    n_max: int = 2000 if nb.MODE == "light" else 5000
cfg = Cfg()
cfg


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
source_path = nb.ROOT / 'reports/is_high_risk_auc.json'
source = None
if source_path.exists():
    try:
        if source_path.suffix == ".log":
            source = source_path.read_text(errors="replace")
            print(f"[evidence loaded] {source_path}")
        elif source_path.name == "privacy_state.json":
            source = json.loads(source_path.read_text())
            summary = json.loads((nb.ROOT/"docs/evidence/c6_run_2026-11-21/run_summary.json").read_text())
            print(f"[state loaded] {source_path}; run provenance:", {k:summary["run"].get(k) for k in ("git_sha","git_dirty","config_hash","dataset_version")})
        else:
            source = nb.load_artifact(source_path)
            print(f"[artifact loaded] {source_path}")
    except Exception as exc:
        print(f"[artifact skipped] {source_path}: {type(exc).__name__}: {exc}")
else:
    print(f"[artifact skipped] missing {source_path}")
records = []
if source is not None:
    bayes=source["test_split_bayes"]["per_client"]
    for client,row in bayes.items():
        model=row["models"]["federated_median_test_auc"]
        records.append((client,row["bayes_test_auc"],model["value"],row["ci95_low"],row["ci95_high"]))
    print("Bayes method:",source["test_split_bayes"]["method"])

frame = pd.DataFrame(records, columns=["label", "x", "value", "lower", "upper"])
print("source rows:", len(frame))
frame.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
if frame.empty: ax.text(.5,.5,"No measured artifact values available",ha="center",va="center",transform=ax.transAxes)
else: ax.bar(frame["label"].astype(str), frame["value"].astype(float), color="#2a6f97")
ax.set_title("Measured values by source row"); ax.set_ylabel("Value"); ax.tick_params(axis="x",rotation=30)
fig.tight_layout(); nb.save_fig(fig,"test_split_bayes_sanity_bar"); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
if frame.empty: ax.text(.5,.5,"No measured artifact values available",ha="center",va="center",transform=ax.transAxes)
else: ax.scatter(frame["x"].astype(float), frame["value"].astype(float), s=55, color="#d1495b")
ax.set_title("Measured values against the recorded x variable"); ax.set_xlabel("Recorded x"); ax.set_ylabel("Value")
fig.tight_layout(); nb.save_fig(fig,"test_split_bayes_sanity_scatter"); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
values = pd.to_numeric(frame["value"], errors="coerce").dropna() if not frame.empty else pd.Series(dtype=float)
if values.empty: ax.text(.5,.5,"No measured artifact values available",ha="center",va="center",transform=ax.transAxes)
else: ax.hist(values, bins=min(10,max(1,len(values))), color="#edae49", edgecolor="white")
ax.set_title("Distribution of measured values"); ax.set_xlabel("Value"); ax.set_ylabel("Row count")
fig.tight_layout(); nb.save_fig(fig,"test_split_bayes_sanity_hist"); plt.show()


**How to read this / what it shows.** This histogram summarizes only the available source rows. A missing-artifact message is shown instead of substituting sample values.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
if frame.empty: ax.text(.5,.5,"No measured artifact values available",ha="center",va="center",transform=ax.transAxes)
else:
    ordered = frame.sort_values("x")
    ax.plot(ordered["x"].astype(float), ordered["value"].astype(float), marker="o", color="#00798c")
ax.set_title("Measured values in source order"); ax.set_xlabel("Recorded x"); ax.set_ylabel("Value")
fig.tight_layout(); nb.save_fig(fig,"test_split_bayes_sanity_line"); plt.show()


**How to read this / what it shows.** The line connects source rows sorted by their recorded x value to show ordering or trend within this artifact; it is not an extrapolation.

## Limitations and failure cases
- This is a synthetic, local evidence review; it is not production or multi-organization evidence.
- If the source artifact is missing or fails its provenance check, the notebook prints a skip message and its plots remain empty.
- The displayed summaries do not add privacy protection to their source records.

## References
- Project API and evidence source: `reports/is_high_risk_auc.json`.
- Shared notebook helpers: `notebooks/_nb_common.py`.
